# 第15讲 · 神经网络原理与 BP 非线性拟合 —— 1-16-1 numpy 手写

本 notebook 对应课件 `lecture15.md` 与脚本 `neural_net.py`。

**一句话目标**：线性回归只能画一条直线，而神经网络能拟合任意曲线——这里用**手写**的反向传播（BP）拟合 `y = 0.5·sin(2u) + 0.3u`。

完整流程：
1. 造曲线状数据
2. 手写 3 层 MLP(1→16→1, tanh)
3. 前向 / 反向 / 更新循环训练 5000 轮
4. 对比线性回归与神经网络的 MSE
5. 观察损失曲线是否一路下降

依赖：`numpy`、`scikit-learn`（仅线性回归对照）、`matplotlib`。

In [ ]:
# 让绘图直接嵌入 notebook（不弹窗）
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from plots import setup_zh_font

# 中文字体设置
setup_zh_font()

## 1. 造曲线数据

在 `u ∈ [-2, 2]` 上取 200 个点，目标函数是非线性曲线 `y = 0.5·sin(2u) + 0.3u`——线性回归只能近似成直线，这正是神经网络要超越它的地方。

In [ ]:
rng = np.random.default_rng(42)
u = np.linspace(-2, 2, 200).reshape(-1, 1)
y_true = 0.5 * np.sin(2 * u) + 0.3 * u
print(f"输入 shape={u.shape}, 目标 shape={y_true.shape}")

## 2. 手写 3 层 MLP(1→16→1, tanh)

网络结构：**输入层 1 个神经元 → 隐藏层 16 个 → 输出层 1 个**。

- **He 初始化**：`w1` 用 `sqrt(2.0)` 缩放，`w2` 用 `sqrt(2.0/n_hidden)`，帮助训练稳定（课件代码沿用的经验值）；截距初始为 0。
- **前向传播** `forward`：`z1 = X@w1+b1` → `a1 = tanh(z1)`（激活函数引入非线性）→ `a1@w2+b2`。
- **反向传播** `backward`：算输出误差 → 沿链式法则回传到隐藏层（tanh 导数 = 1-tanh²）→ 梯度下降更新两组权重与偏置。

In [ ]:
class MLP:
    """3 层网络 1 -> n_hidden -> 1，tanh 激活，随机梯度下降更新。"""
    def __init__(self, n_hidden=16, lr=0.001):
        # He 初始化，截距初始为 0
        self.w1 = np.random.randn(1, n_hidden) * np.sqrt(2.0)
        self.b1 = np.zeros((1, n_hidden))
        self.w2 = np.random.randn(n_hidden, 1) * np.sqrt(2.0 / n_hidden)
        self.b2 = np.zeros((1, 1))
        self.lr = lr

    def forward(self, X):
        z1 = X @ self.w1 + self.b1
        self.a1 = np.tanh(z1)                # 激活函数引入非线性
        return self.a1 @ self.w2 + self.b2

    def backward(self, X, y):
        """一次『前向算误差 → 反向传播 → 梯度下降更新』，返回前向预测。"""
        y_pred = self.forward(X)
        dz2 = y_pred - y                     # 输出层误差
        # 更新输出层
        self.w2 -= self.lr * (self.a1.T @ dz2)
        self.b2 -= self.lr * dz2.sum(axis=0)
        # 误差反向传播到隐藏层（tanh 导数 = 1 - tanh²）
        dz1 = (dz2 @ self.w2.T) * (1 - self.a1 ** 2)
        self.w1 -= self.lr * (X.T @ dz1)
        self.b1 -= self.lr * dz1.sum(axis=0)
        return y_pred


def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

## 3. 线性回归对照（只能画一条直线）

先用 `LinearRegression` 拟合曲线数据——它只能得到一条直线，我们将以此与神经网络对比 MSE。

In [ ]:
from sklearn.linear_model import LinearRegression
lin = LinearRegression().fit(u, y_true)
y_lin = lin.predict(u)
mse_lin = mse(y_true, y_lin)
print(f"线性回归 均方误差 = {mse_lin:.4f}")

## 4. 训练神经网络（手写 BP，5000 轮）

每轮调用 `net.backward`（前向误差 → 反向传播 → 更新），每 500 轮记录一次损失，便于观察是否一路下降。

In [ ]:
net = MLP(n_hidden=16, lr=0.001)
losses = []
epochs = 5000
for epoch in range(epochs):
    y_pred = net.backward(u, y_true)
    if epoch % 500 == 0:
        losses.append((epoch, mse(y_true, y_pred)))
y_nn = net.forward(u)
mse_nn = mse(y_true, y_nn)

print("=== 神经网络(1-16-1, tanh) vs 线性回归 ====")
print(f"线性回归 均方误差 = {mse_lin:.4f}")
print(f"神经网络 均方误差 = {mse_nn:.5f}  （小了 {mse_lin/mse_nn:.0f} 倍）")
print("\n  训练过程损失（每 500 轮采样）：")
for ep, l in losses:
    arrow = " ←" if ep == 0 else ""
    print(f"    epoch {ep:>4}: loss={l:.5f}{arrow}")
print("  损失应一路下降 → 训练有效；若震荡或 nan → 学习率过大。")

## 5. 可视化：拟合对比 + 损失曲线

左图：真实曲线 vs 线性回归直线 vs 神经网络拟合——神经网络能弯曲贴合。

右图：训练过程损失曲线——应一路下降。

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1)
plt.plot(u, y_true, "k-", label="真实曲线")
plt.plot(u, y_lin, "r--", label=f"线性回归 (MSE={mse_lin:.3f})")
plt.plot(u, y_nn, "b-", label=f"神经网络 (MSE={mse_nn:.3f})")
plt.title("非线性拟合对比")
plt.xlabel("u"); plt.ylabel("y")
plt.legend(); plt.grid(alpha=0.4)

plt.subplot(1, 2, 2)
eps = [e for e, _ in losses]
lss = [l for _, l in losses]
plt.plot(eps, lss, "g-")
plt.title("神经网络损失曲线")
plt.xlabel("epoch"); plt.ylabel("MSE")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## 上机挑战与提示

- ⚠️ 学习率 `> ~0.01` 容易发散（loss→nan），本 notebook 用默认 0.001。
- ⚠️ 把 `tanh` 换成 `ReLU` 试试：隐藏层激活改 `relu`，回传导数相应改为 `1(正)/0(负)`。

试着改改 `lr` 或激活函数，看看损失曲线和拟合效果怎么变。